# Analysis of samples from TNP-KR

## TODO

$$
\def\ctx{\mathrm{ctx}}
\def\test{\mathrm{test}}
\def\autoreg{\mathrm{autoreg}}
$$

- plot difference heatmaps
- plot monte carlo traces (do things even converge?)
- forward error? i.e. distribution of $q_\theta^\autoreg(\test | \ctx)p(\ctx)$ vs $p(\ctx, \test)$ vs $q_\theta(\test | \ctx)p(\ctx)$


## Setup

In [ ]:
from pathlib import Path
from typing import Callable

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy.stats as stats
from numpy.typing import NDArray
from omegaconf import OmegaConf
from sps.kernels import rbf

plt.style.use("default")
results_dir = Path(
    "/Users/pgrynfelder/Library/CloudStorage/GoogleDrive-wadh6460@ox.ac.uk/My Drive/results"
)


runs = sorted(
    list(set(map(lambda x: x.parent, results_dir.glob("**/*.npy")))),
    key=lambda path: path.stat().st_mtime,
)

In [ ]:
# examples where the analytic solution exists: 1, 2, 4, 5, 6, 8, 9, 10, 12, 13, 14, 16, 17, 18
# nice examples: 4,
# run = results_dir / "13"

prompt = "Which run? Type a number.\n" + "\n".join(
    [f"{i}: {run.name}" for i, run in enumerate(runs)]
)
print(prompt)
i = int(input(prompt))
run = runs[i]
print(f"Selected {run.name}")

# Load data
data = dict()
for file in run.glob("*.npy"):
    data[file.stem] = np.load(file).squeeze()  # simplify dimensionality

# Sort s_test
idx = np.argsort(data["s_test"])
data["s_test"] = data["s_test"][idx]
data["f_test"] = data["f_test"][idx]
for key in data.keys():
    if key.startswith("paths_"):
        data[key] = data[key][:, idx]
data["diagonal_mu"] = data["diagonal_mu"][idx]
data["diagonal_var"] = data["diagonal_var"][idx]

print("Seed:", data["seed"])
for key, value in sorted(data.items()):
    print(key, value.shape)

if (run / "config.yaml").exists():
    config = OmegaConf.load(run / "config.yaml")
else:
    config = None

print("Config:", config)

paths = sorted([key for key in data.keys() if "paths_" in key])
if "paths_diagonal" in paths:
    paths.remove("paths_diagonal")  # no need to see this for most plots
strategies = [key.lstrip("paths_") for key in paths]

summary = pd.DataFrame()

In [ ]:
from json import dumps

print(dumps(OmegaConf.to_object(config), indent=4))
print("Variance", data["var"])
print("Lengthscale", data["ls"])


In [ ]:
# Utils
from jax.experimental import enable_x64


def grid(num_items: int, ncols: int = 3, **kwargs):
    """`plt.subplots` with better defaults"""
    nrows = (num_items - 1) // ncols + 1
    figsize = (ncols * 5, nrows * 5)

    fig, axes = plt.subplots(nrows, ncols, figsize=figsize, **kwargs)
    fig.set_constrained_layout(True)
    for ax in axes.flat[num_items:]:
        ax.axis("off")
    return fig, axes


def logpdf(s, f, kernel, var, ls):
    s = s.astype(np.float64)
    f = f.astype(np.float64)
    with enable_x64():
        cov = kernel(s, s, var, ls)

    return stats.multivariate_normal.logpdf(f, cov=cov)

## Analytic solution

Note that it assumes we know the kernel, variance, lengthscale, and observation noise so requires strictly more information than the model.

In [ ]:
from dl4bi.meta_learning.autoregressive import analytic_gp

analytic_mu, analytic_cov = analytic_gp(
    data["s_ctx"],
    data["f_ctx"],
    data["s_test"],
    kernel=rbf,
    var=data["var"],
    ls=data["ls"],
    # noise=config.data.obs_noise,
    noise=1e-8,
)
analytic_mu, analytic_cov = np.array(analytic_mu), np.array(analytic_cov)

data["analytic_mu"] = analytic_mu
data["analytic_cov"] = analytic_cov

print("Analytic mu", analytic_mu.shape)
print("Analytic cov", analytic_cov.shape)

## Metrics

### L2 distance from analytical solution

The $L^2$ squared distance between continuous distributions is given by

$$ \int |p(x) - q(x)|^2 \mathrm{d} x .$$

As we have samples from $q$ available along with 'densities' but can't access the pdf directly, we will use an importance sampling estimate:

$$ \def\E{\mathbb{E}}
\widehat\E_{x \sim q}[|p(x) - q(x)|^2 / q(x)]
$$

Note this is only accurate if both distributions have the same support.

In [ ]:
def L2_estimate(
    paths: NDArray,  # [Np, L_test]
    log_densities: NDArray,  # [Np]
    s_test: NDArray,  # [L_test]
    s_ctx: NDArray,  # [L_ctx]
    f_ctx: NDArray,  # [L_ctx]
    kernel: Callable,
    var: float,
    ls: float,
    jitter=0,
    running=False,  # whether to return a running estimate
):
    Np = paths.shape[0]

    s_ctx = s_ctx.astype(np.float64)
    s_test = s_test.astype(np.float64)
    f_ctx = f_ctx.astype(np.float64)

    # need to call the kernel function in jax 64-bit precision context
    with enable_x64():
        s = np.concatenate([s_ctx, s_test])
        K_joint = kernel(s, s, var, ls) + jitter * np.eye(len(s))
        dist_joint = stats.multivariate_normal(cov=K_joint, allow_singular=True)

        K_ctx = kernel(s_ctx, s_ctx, var, ls) + jitter * np.eye(len(s_ctx))
        logp_context = stats.multivariate_normal.logpdf(
            f_ctx, cov=K_ctx, allow_singular=True
        )

    samples = []
    for path, logq in zip(paths, log_densities):
        f = np.concatenate([f_ctx, path])
        logp_joint = dist_joint.logpdf(f)

        samples.append(
            (np.exp(logq) - np.exp(logp_joint - logp_context)) ** 2 / np.exp(logq)
        )

    if running:
        return np.cumsum(samples) / np.arange(1, Np + 1)
    else:
        return np.mean(samples)


for path in paths:
    strategy = path.split("_")[-1]
    estimate = L2_estimate(
        data[path],
        data[f"densities_{strategy}"],
        data["s_test"],
        data["s_ctx"],
        data["f_ctx"],
        rbf,
        data["var"],
        data["ls"],
        jitter=1e-9,
        running=False,
    )
    print(strategy, estimate, sep="\t")

### KL divergence

We can estimate the reverse KL-divergence
$$ 
\mathrm{KL}\left(q^\autoreg(\cdot | \ctx)\,\|\, p(\cdot | \ctx)\right) = \E_{Y_{s_\test} \sim q^\autoreg(\cdot | \ctx)}\left[\log \frac{q^\autoreg(Y_{s_\test} | \ctx)}{p(Y_{s_\test} | \ctx)}\right]
$$
using the autoregressive samples from the model.

In [ ]:
# TODO: this is completely broken. The density p of most paths is ~= 0, which leads to infinite KL divergence.
# What's worse, I suspect the true KL divergence values are indeed expected to be very large and not fit in float64.

from jax.experimental import enable_x64


def KL_estimate(
    paths: NDArray,  # [Np, L_test]
    log_densities: NDArray,  # [Np]
    s_test: NDArray,  # [L_test]
    s_ctx: NDArray,  # [L_ctx]
    f_ctx: NDArray,  # [L_ctx]
    kernel: Callable,
    var: float,
    ls: float,
    jitter=0,
    running=False,  # whether to return a running estimate
):
    """
    Simple Monte Carlo KL(q || posterior) estimate where q induced by model, posterior is the analytically computed posterior.

    Note that post = p(test | ctx) = p(ctx, test) / p(ctx),
    i.e. we can use chain rule to avoid computing the conditional variance.
    """
    Np = paths.shape[0]
    L_ctx = s_ctx.shape[0]
    L_test = s_test.shape[0]

    s_ctx = s_ctx.astype(np.float64)
    s_test = s_test.astype(np.float64)
    f_ctx = f_ctx.astype(np.float64)

    # need to call the kernel function in jax 64-bit precision context
    with enable_x64():
        s = np.concatenate([s_ctx, s_test])
        K_joint = kernel(s, s, var, ls) + jitter * np.eye(L_ctx + L_test)
        dist_joint = stats.multivariate_normal(cov=K_joint, allow_singular=True)

        K_ctx = kernel(s_ctx, s_ctx, var, ls) + jitter * np.eye(L_ctx)
        logp_context = stats.multivariate_normal.logpdf(
            f_ctx, cov=K_ctx, allow_singular=True
        )

    samples = []
    for path, logq in zip(paths, log_densities):
        f = np.concatenate([f_ctx, path])
        logp_joint = dist_joint.logpdf(f)

        if np.abs(logp_joint) < 1e6:
            print(f"Warning, large logp_joint! {logp_joint=} {logp_context=} {logq=}")

        samples.append(logq - (logp_joint - logp_context))

    if running:
        return np.cumsum(samples) / np.arange(1, Np + 1)
    else:
        return np.mean(samples)


maxlen = max(map(len, paths + ["diagonal"]))
fig, axes = grid(len(paths))
for path, ax in zip(paths, axes.flat):
    strategy = path.split("_")[-1]
    running_kl = KL_estimate(
        data[path],
        data[f"densities_{strategy}"],
        data["s_test"],
        data["s_ctx"],
        data["f_ctx"],
        rbf,
        data["var"],
        data["ls"],
        jitter=1e-5,
        running=True,
    )

    indices = np.arange(100, len(running_kl))  # drop first 100 steps from the plot
    ax.plot(indices + 1, running_kl[indices])
    ax.text(
        0.9,
        0.9,
        f"KL={running_kl[-1]:.2f}",
        transform=ax.transAxes,
        ha="right",
        va="top",
    )
    summary.loc[path, "KL"] = running_kl[-1]

    ax.set_title(f"{path}")
    ax.set_xlabel("Number of sample paths")

fig.suptitle("Estimating KL(q || posterior) for different path sampling strategies")
plt.show()


print(summary.KL.sort_values())

## Covariance

In [ ]:
images = {
    "analytic": data["analytic_cov"],
    "diagonal": np.diag(data["diagonal_var"]),
}
for key in paths:
    images[key] = np.cov(data[key].T)

vmin = min([image.min() for image in images.values()])
vmax = max([image.max() for image in images.values()])
v = max(abs(vmin), abs(vmax))

plt.tight_layout()
cmap = plt.get_cmap("RdBu")
norm = plt.Normalize(vmin=-v, vmax=v)
im = plt.cm.ScalarMappable(norm=norm, cmap=cmap)

fig, axes = grid(len(images), sharex=True, sharey=True)
for (path, image), ax in zip(images.items(), axes.flat):
    ax.imshow(image, cmap=cmap, norm=norm)

    # Draw where context points are.
    # This assumes s_test is a linspace.
    # ctx = (
    #     (data["s_ctx"] + data["s_test"].max())
    #     / (data["s_test"].max() - data["s_test"].min())
    #     * len(data["s_test"])
    # )
    ctx = [np.searchsorted(data["s_test"], s) for s in data["s_ctx"]]
    ax.hlines(ctx, *ax.get_xlim(), "g", linewidth=0.3)
    ax.vlines(ctx, *ax.get_ylim(), "g", linewidth=0.3)

    ax.set_title(path)
    ax.axis("off")
fig.colorbar(im, ax=axes.flatten())
fig.suptitle(
    "Covariance matrices for different path sampling strategies + analytic solutions"
)
plt.show()


for path, image in images.items():
    summary.loc[path, "Covariance - Frobenius distance from analytic"] = np.linalg.norm(
        image - images["analytic"], ord="fro"
    )

print(summary["Covariance - Frobenius distance from analytic"].sort_values())

## Mean

In [ ]:
means = {
    "analytic": data["analytic_mu"],
    "diagonal": data["diagonal_mu"],
} | {path: np.mean(data[path], axis=0) for path in paths}

fig, axes = grid(len(means))
for (path, mean), ax in zip(means.items(), axes.flat):
    ax.set_title(path)
    ax.plot(data["s_test"], mean)
    # ax.plot(data["s_test"], data["analytic_mu"], "--", label="Analytic", alpha=0.5)
    ax.plot(data["s_ctx"], data["f_ctx"], "g+", label="Context")

fig.suptitle("$\mu$")
plt.legend()
plt.plot()

## Variance

In [ ]:
variances = {
    "analytic": np.diag(data["analytic_cov"]),
    "diagonal": data["diagonal_var"],
} | {path: np.var(data[path], axis=0) for path in paths}


fig, axes = grid(len(variances))
for (path, variance), ax in zip(variances.items(), axes.flat):
    ax.set_title(path)
    ax.plot(data["s_test"], variance)
    # ax.plot(
    #     data["s_test"],
    #     np.diag(data["analytic_cov"]),
    #     "--",
    #     alpha=0.5,
    #     label="Analytic variance",
    # )
    ax.plot(data["s_ctx"], 0 * data["s_ctx"], "g+", label="Context")
fig.set_constrained_layout(True)
fig.suptitle("Variance")
plt.legend()
plt.show()

## Means + 95% CI

In [ ]:
fig, axes = grid(len(means))

for path, ax in zip(means.keys(), axes.flat):
    mu = means[path]
    var = variances[path]
    lo = mu - 2 * np.sqrt(var)
    hi = mu + 2 * np.sqrt(var)
    ax.plot(data["s_test"], means[path], alpha=0.5)
    ax.fill_between(data["s_test"], lo, hi, alpha=0.2)
    ax.plot(data["s_ctx"], data["f_ctx"], "g.", label="Context", linewidth=500)
    # ax.plot(data["s_test"], data["analytic_mu"], "--", label="analytic mean", alpha=0.5)
    ax.legend()
    ax.set_title(path)

fig.suptitle("$\mu \pm 2\sigma$")
plt.show()

## Example sample paths

In [ ]:
Np = 4

indices = np.random.choice(data["paths_ltr"].shape[0], (Np,), replace=False)

fig, axes = grid(len(paths) * Np, ncols=Np, sharey=True, sharex=True)
for path, axs in zip(paths, axes):
    axs[0].set_ylabel(path)
    for idx, ax in zip(indices, axs):
        ax.plot(data["s_test"], data[path][idx])
        ax.plot(data["s_ctx"], data["f_ctx"], "g.", label="Context")
fig.suptitle("Autoregressive sample paths")

plt.show()

In [ ]:
plt.title("A path from the true gp")
plt.plot(data["s_test"], data["f_test"])
plt.plot(data["s_ctx"], data["f_ctx"], "g.", label="Context")

## Estimation traces

In [ ]:
path = "paths_ltr"
N_locations = 100
ns = np.arange(11, data[path].shape[0] + 1)  # dropping first 10 steps from the plot

np.random.seed(0)
locations = np.random.choice(data["s_test"].shape[0], N_locations, replace=False)
locations.sort()

In [ ]:
# plot running means

fig, axes = grid(N_locations, ncols=5)
for location, ax in zip(locations, axes.flat):
    running_sum = data[path][:, location].cumsum()[ns - 1]
    running_mean = running_sum / ns
    ax.plot(ns, running_mean, label="Running mean", linewidth=2)
    ax.set_title(f"$s = {data['s_test'][location]:.2f}$")

fig.suptitle(f"Running means for {path} at {N_locations} randomly chosen locations")
plt.show()


### plot running variances

In [ ]:
ns = np.arange(
    11, data[path].shape[0] + 1
)  # note dropping the first 10 steps from the plots
running_variances = np.stack([np.var(data[path][:n, locations], axis=0) for n in ns]).T

fig, axes = grid(N_locations, ncols=5)
for location, ax, running_variance in zip(locations, axes.flat, running_variances):
    ax.plot(ns, running_variance)
    ax.set_title(f"$s = {data['s_test'][location]:.2f}$")

fig.suptitle(f"Running variances for {path}")
plt.show()

### plot histograms

In [ ]:
fig, axes = grid(N_locations, ncols=5)
for location, ax in zip(locations, axes.flat):
    ax.hist(data[path][:, location], bins="auto", density=True)
    ax.set_title(f"$s = {data['s_test'][location]:.2f}$")
fig.suptitle(
    f"Histograms of value of {path} at {N_locations} randomly chosen locations"
)
plt.plot()

### plot qq plots

In [ ]:
fig, axes = grid(N_locations, ncols=4)
for location, ax in zip(locations, axes.flat):
    _, (slope, intercept, _) = stats.probplot(
        data[path][:, location], dist="norm", plot=ax
    )
    ax.set_title(f"$s = {data['s_test'][location]:.2f}$")

fig.suptitle(f"QQ plots of value of {path} at {N_locations} randomly chosen locations")
plt.plot()

### Shapiro-Wilk test for normality

This is expected to fail almost always for large sample sizes.

In [ ]:
statistic, pvalue = stats.shapiro(data[path], axis=0)

alpha = 0.05


print(
    f"{(pvalue < alpha).sum()} out of {len(pvalue)} are not normal according to Shapiro-Wilk test with alpha={alpha}"
)

In [ ]:
%pip install mvshapirotest mvem

In [ ]:
from mvShapiroTest.test import mvshapiro

mvshapiro(data[path] / 100)